# 11_xgboost_balanced.ipynb

This notebook tests a different modeling approach on the Titanic dataset.


In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier, VotingClassifier, StackingClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

y = train['Survived']
train_ids = train['PassengerId'].copy()
test_ids = test['PassengerId'].copy()

def feature_engineering(df):
    df = df.copy()

    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    df['Mother'] = ((df['Sex'] == 'female') & (df['Parch'] > 0) & (df['Age'] > 18) & (df['Parch'] < 5)).astype(int)
    df['Child'] = (df['Age'] < 14).astype(int)
    df['AgeMissing'] = df['Age'].isna().astype(int)
    df['FareMissing'] = df['Fare'].isna().astype(int)
    df['EmbarkedMissing'] = df['Embarked'].isna().astype(int)
    df['HasCabin'] = df['Cabin'].notna().astype(int)
    df['CabinDeck'] = df['Cabin'].fillna('U').str[0]

    df['Title'] = df['Name'].str.extract(r',\\s*([^.]*)\\.', expand=False).str.strip()
    df['Title'] = df['Title'].replace({'Mlle':'Miss', 'Ms':'Miss', 'Mme':'Mrs'})
    common_titles = ['Mr', 'Miss', 'Mrs', 'Master']
    df.loc[~df['Title'].isin(common_titles), 'Title'] = 'Rare'

    df['Surname'] = df['Name'].str.split(',').str[0].str.strip()
    df['TicketPrefix'] = df['Ticket'].str.replace(r'\\d', '', regex=True).str.replace(r'[./]', '', regex=True).str.replace(' ', '', regex=True).replace('', 'NONE')
    df['TicketGroupSize'] = df.groupby('Ticket')['Ticket'].transform('count')
    df['SurnameGroupSize'] = df.groupby('Surname')['Surname'].transform('count')
    df['FarePerPerson'] = df['Fare'] / df['TicketGroupSize'].replace(0, 1)
    df['SexPclass'] = df['Sex'].astype(str) + '_' + df['Pclass'].astype(str)
    df['FamilySizeBand'] = pd.cut(df['FamilySize'], bins=[0,1,4,7,100], labels=['Alone','Small','Medium','Large'])
    df['AgeBand'] = pd.cut(df['Age'], bins=[-1,5,12,18,30,45,60,100], labels=['Baby','Child','Teen','YoungAdult','Adult','MiddleAge','Senior'])
    df['FareBand'] = pd.qcut(df['Fare'].rank(method='first'), 5, labels=['VeryLow','Low','Medium','High','VeryHigh'])
    df['FamilySex'] = df['Sex'].astype(str) + '_' + df['FamilySizeBand'].astype(str)
    df['PclassTitle'] = df['Pclass'].astype(str) + '_' + df['Title'].astype(str)
    df['PclassAgeBand'] = df['Pclass'].astype(str) + '_' + df['AgeBand'].astype(str)
    df['FamilyTicket'] = df['FamilySize'].astype(str) + '_' + df['TicketPrefix'].astype(str)
    df['FarePerPersonMissing'] = df['FarePerPerson'].isna().astype(int)
    df['NameLength'] = df['Name'].str.len()
    df['NameWords'] = df['Name'].str.split().str.len()
    df['TicketLength'] = df['Ticket'].str.len()
    df['CabinCount'] = df['Cabin'].fillna('').str.split().str.len()
    df['DeckKnown'] = (df['CabinDeck'] != 'U').astype(int)
    df['LargeFamily'] = (df['FamilySize'] >= 5).astype(int)
    df['SmallFamily'] = df['FamilySize'].between(2, 4).astype(int)
    df['FemaleChild'] = ((df['Sex'] == 'female') | (df['Age'] < 14)).astype(int)
    df['FarePerAge'] = df['Fare'] / (df['Age'].fillna(df['Age'].median()) + 1)
    df['ClassFare'] = df['Fare'] / df['Pclass']
    df['SiblingChildRatio'] = df['SibSp'] / (df['Parch'] + 1)
    df['FamilyFare'] = df['Fare'] * df['FamilySize']
    df['SexTitle'] = df['Sex'].astype(str) + '_' + df['Title'].astype(str)
    
    return df

train_fe = feature_engineering(train.drop(columns=['Survived']))
test_fe = feature_engineering(test.copy())

drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin', 'Surname']
X = train_fe.drop(columns=drop_cols, errors='ignore')
X_test = test_fe.drop(columns=drop_cols, errors='ignore')

numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(exclude=['number']).columns.tolist()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent'))
    ,('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

model = XGBClassifier(n_estimators=500, max_depth=3, learning_rate=0.03, subsample=0.85, colsample_bytree=0.85, min_child_weight=2, reg_alpha=0.05, reg_lambda=1.5, objective='binary:logistic', eval_metric='logloss', random_state=42, n_jobs=-1)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
])

pipeline.fit(X_train, y_train)
valid_pred = pipeline.predict(X_valid)
score = accuracy_score(y_valid, valid_pred)

print(f'Validation accuracy: {score:.4f}')


Validation accuracy: 0.8101
